In [0]:
%pip install fpdf2 faker -q
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import os

# Determinar diretório atual dinamicamente via contexto do notebook
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = '/Workspace' + os.path.dirname(notebook_path)

base_path = os.path.join(notebook_dir, 'petroquimica_dataset')
contratos = pd.read_csv(f"{base_path}/contratos.csv")
fornecedores = pd.read_csv(f"{base_path}/fornecedores.csv")
ordens = pd.read_csv(f"{base_path}/ordens_servico.csv")

# Merge contratos com fornecedores
contratos_full = contratos.merge(fornecedores, on='id_fornecedor', how='left')

# Diretório de saída = diretório atual do notebook no workspace
pdf_contratos_dir = os.path.join(notebook_dir, 'pdfs_contratos')
pdf_procedimentos_dir = os.path.join(notebook_dir, 'pdfs_procedimentos')

os.makedirs(pdf_contratos_dir, exist_ok=True)
os.makedirs(pdf_procedimentos_dir, exist_ok=True)

print(f"Contratos carregados: {len(contratos_full)}")
print(f"Fornecedores carregados: {len(fornecedores)}")
print(f"Ordens carregadas: {len(ordens)}")
print(f"PDFs de contratos serão salvos em: {pdf_contratos_dir}")
print(f"PDFs de procedimentos serão salvos em: {pdf_procedimentos_dir}")

Contratos carregados: 1000
Fornecedores carregados: 100
Ordens carregadas: 500
PDFs de contratos serão salvos em: /Workspace/Users/carolina.ferreira@databricks.com/pdfs_contratos
PDFs de procedimentos serão salvos em: /Workspace/Users/carolina.ferreira@databricks.com/pdfs_procedimentos


In [0]:
from faker import Faker

fake = Faker('pt_BR')
Faker.seed(42)

# Dados ficticios da empresa contratante
EMPRESA_NOME = fake.company()
EMPRESA_NOME_COMPLETO = f"Fake Company S.A."
EMPRESA_NOME_UPPER = EMPRESA_NOME_COMPLETO.upper()
EMPRESA_CNPJ = fake.cnpj()
EMPRESA_RUA = f"{fake.street_name()}, {fake.building_number()}"
EMPRESA_CIDADE = fake.city()
EMPRESA_ESTADO_SIGLA = fake.state_abbr()
EMPRESA_ESTADO = fake.state()
EMPRESA_ENDERECO_CURTO = f"{EMPRESA_RUA} - Complexo Industrial de {EMPRESA_CIDADE} - {EMPRESA_ESTADO_SIGLA}"
EMPRESA_ENDERECO_COMPLETO = f"{EMPRESA_RUA}, Complexo Industrial de {EMPRESA_CIDADE}, {EMPRESA_CIDADE}/{EMPRESA_ESTADO_SIGLA}"

print(f"Empresa ficticia gerada: {EMPRESA_NOME_COMPLETO}")
print(f"CNPJ: {EMPRESA_CNPJ}")
print(f"Endereco: {EMPRESA_ENDERECO_CURTO}")

Empresa ficticia gerada: Fake Company S.A.
CNPJ: 43.815.092/0001-70
Endereco: Área Bento Borges, 93 - Complexo Industrial de Novais - DF


In [0]:
from fpdf import FPDF
import textwrap

EFF_W = 190  # largura efetiva A4 com margens de 10mm

def _clean(text):
    """Remove caracteres fora do Latin-1 para compatibilidade com Helvetica."""
    replacements = {'\u2022': '-', '\u2013': '-', '\u2014': '-', '\u201c': '"', '\u201d': '"', '\u2018': "'", '\u2019': "'", '\u2026': '...'}
    for k, v in replacements.items():
        text = text.replace(k, v)
    return text.encode('latin-1', errors='replace').decode('latin-1')


class ContratoPDF(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 10)
        self.cell(0, 6, _clean(EMPRESA_NOME_UPPER), align='C', new_x='LMARGIN', new_y='NEXT')
        self.set_font('Helvetica', '', 8)
        self.cell(0, 5, _clean(f'CNPJ: {EMPRESA_CNPJ}'), align='C', new_x='LMARGIN', new_y='NEXT')
        self.cell(0, 5, _clean(EMPRESA_ENDERECO_CURTO), align='C', new_x='LMARGIN', new_y='NEXT')
        self.line(10, self.get_y() + 2, 200, self.get_y() + 2)
        self.ln(6)

    def footer(self):
        self.set_y(-20)
        self.set_font('Helvetica', 'I', 7)
        self.cell(0, 5, 'Documento Confidencial - Uso Interno', align='C', new_x='LMARGIN', new_y='NEXT')
        self.cell(0, 5, f'Pagina {self.page_no()}/{{nb}}', align='C')


def _write_section(pdf, title, body):
    """Escreve uma secao do contrato de forma robusta."""
    pdf.set_x(10)
    pdf.set_font('Helvetica', 'B', 10)
    pdf.multi_cell(EFF_W, 8, _clean(title))
    pdf.set_font('Helvetica', '', 9)
    for line in body.split('\n'):
        line = line.strip()
        if line:
            pdf.set_x(10)
            pdf.multi_cell(EFF_W, 5, _clean(line))
        else:
            pdf.ln(3)
    pdf.ln(4)


def _valor_por_extenso(valor):
    if valor >= 1_000_000:
        return f"aproximadamente {valor/1_000_000:.1f} milhoes de reais"
    elif valor >= 1_000:
        return f"aproximadamente {valor/1_000:.0f} mil reais"
    else:
        return f"{valor:.2f} reais"


def gerar_contrato_pdf(row, output_path):
    pdf = ContratoPDF()
    pdf.alias_nb_pages()
    pdf.set_auto_page_break(auto=True, margin=25)
    pdf.add_page()

    # Titulo
    pdf.set_font('Helvetica', 'B', 14)
    pdf.cell(0, 10, _clean('CONTRATO DE PRESTACAO DE SERVICOS'), align='C', new_x='LMARGIN', new_y='NEXT')
    pdf.set_font('Helvetica', 'B', 11)
    pdf.cell(0, 8, _clean(f"No {row['numero_contrato']}"), align='C', new_x='LMARGIN', new_y='NEXT')
    pdf.ln(3)
    pdf.set_font('Helvetica', 'B', 9)
    pdf.cell(0, 6, _clean(f"Tipo: {row['tipo_contrato']}"), new_x='LMARGIN', new_y='NEXT')
    pdf.cell(0, 6, _clean(f"Unidade Industrial: {row['unidade_industrial']}"), new_x='LMARGIN', new_y='NEXT')
    pdf.cell(0, 6, _clean(f"Area Responsavel: {row['area_responsavel']}"), new_x='LMARGIN', new_y='NEXT')
    pdf.ln(4)

    contratada = str(row.get('razao_social', 'Fornecedor nao identificado'))
    cnpj_forn = str(row.get('cnpj', '00.000.000/0001-00'))
    cidade_forn = str(row.get('cidade', 'Sao Paulo'))
    estado_forn = str(row.get('estado', 'SP'))

    _write_section(pdf, 'CLAUSULA 1a - DAS PARTES',
        f"CONTRATANTE: {EMPRESA_NOME_COMPLETO}, pessoa juridica de direito privado, inscrita no CNPJ sob o no {EMPRESA_CNPJ}, com sede na {EMPRESA_ENDERECO_COMPLETO}, neste ato representada por seus representantes legais.\n\nCONTRATADA: {contratada}, inscrita no CNPJ sob o no {cnpj_forn}, com sede em {cidade_forn}/{estado_forn}, doravante denominada CONTRATADA.")

    _write_section(pdf, 'CLAUSULA 2a - DO OBJETO',
        f"O presente contrato tem por objeto a contratacao de {row['tipo_contrato'].lower()}, conforme especificacoes tecnicas descritas no Anexo I deste instrumento, a ser executado na {row['unidade_industrial']}.\n\nEscopo detalhado: {row['descricao_escopo']}.\n\nOs servicos deverao ser executados em conformidade com as normas tecnicas aplicaveis, incluindo normas ABNT, NRs do Ministerio do Trabalho e regulamentacoes da ANP.")

    valor = row['valor_total']
    moeda = row.get('moeda', 'BRL')
    _write_section(pdf, 'CLAUSULA 3a - DO VALOR E CONDICOES DE PAGAMENTO',
        f"O valor total do presente contrato e de {moeda} {valor:,.2f} ({_valor_por_extenso(valor)}), incluindo todos os custos diretos e indiretos, tributos, encargos sociais, trabalhistas e previdenciarios.\n\nParagrafo 1o: O pagamento sera efetuado em parcelas mensais, mediante apresentacao de medicao aprovada pela fiscalizacao da CONTRATANTE, no prazo de 30 (trinta) dias apos o ateste da Nota Fiscal.\n\nParagrafo 2o: As medicoes mensais deverao ser acompanhadas de relatorio de avanco fisico-financeiro, conforme modelo estabelecido no Anexo III.\n\nParagrafo 3o: A CONTRATANTE retera 5%% (cinco por cento) do valor de cada medicao a titulo de garantia contratual, sendo o valor retido liberado apos a conclusao integral dos servicos e emissao do Termo de Aceite Definitivo.")

    _write_section(pdf, 'CLAUSULA 4a - DA VIGENCIA',
        f"O presente contrato tera vigencia de {row['data_inicio']} a {row['data_fim']}, podendo ser prorrogado mediante Termo Aditivo, desde que haja interesse de ambas as partes e manifestacao formal com antecedencia minima de 60 (sessenta) dias do termino.\n\nParagrafo Unico: O prazo de mobilizacao da CONTRATADA sera de 15 (quinze) dias corridos a partir da emissao da Ordem de Servico inicial.")

    _write_section(pdf, 'CLAUSULA 5a - DAS OBRIGACOES DA CONTRATADA',
        "A CONTRATADA obriga-se a:\n\na) Executar os servicos em conformidade com as especificacoes tecnicas, normas de seguranca e procedimentos operacionais da CONTRATANTE;\nb) Manter equipe tecnica qualificada e devidamente habilitada para a execucao dos servicos;\nc) Fornecer todos os equipamentos de protecao individual (EPI) e coletiva (EPC) necessarios;\nd) Cumprir rigorosamente as normas de Seguranca, Meio Ambiente e Saude (SMS) da CONTRATANTE;\ne) Apresentar documentacao trabalhista e previdenciaria de todos os empregados alocados;\nf) Manter seguro de responsabilidade civil com cobertura compativel com os riscos envolvidos;\ng) Designar preposto para representa-la durante a execucao contratual;\nh) Comunicar imediatamente qualquer incidente, acidente ou anomalia operacional.")

    proc_ref = str(row.get('procedimento_referencia', 'PROC-SMS-001'))
    _write_section(pdf, 'CLAUSULA 6a - SEGURANCA, MEIO AMBIENTE E SAUDE (SMS)',
        f"A CONTRATADA devera cumprir integralmente os procedimentos de SMS da CONTRATANTE, com destaque para o documento de referencia: {proc_ref}.\n\nParagrafo 1o: Antes do inicio das atividades, a CONTRATADA devera realizar integracao de seguranca de todos os colaboradores, incluindo treinamentos especificos sobre:\n- Permissao de Trabalho (PT)\n- Analise Preliminar de Riscos (APR)\n- Procedimentos de emergencia da unidade\n- Manuseio de produtos quimicos perigosos\n- Bloqueio e etiquetagem (LOTO)\n\nParagrafo 2o: O descumprimento das normas de SMS podera ensejar a aplicacao de penalidades previstas neste contrato, incluindo a paralisacao imediata dos servicos.")

    _write_section(pdf, 'CLAUSULA 7a - DAS PENALIDADES',
        "O descumprimento das obrigacoes contratuais sujeitara a CONTRATADA as seguintes penalidades:\n\na) Multa de 0,5%% do valor mensal do contrato por dia de atraso injustificado, limitada a 10%% do valor total;\nb) Multa de 2%% do valor total do contrato por descumprimento de clausula contratual;\nc) Multa de 5%% do valor total em caso de infracao grave as normas de SMS;\nd) Suspensao temporaria de participacao em licitacoes pelo prazo de ate 2 (dois) anos;\ne) Rescisao contratual unilateral, sem prejuizo das demais penalidades.")

    _write_section(pdf, 'CLAUSULA 8a - DA RESCISAO',
        "O presente contrato podera ser rescindido nas seguintes hipoteses:\n\na) Por acordo mutuo entre as partes, mediante notificacao com 30 (trinta) dias de antecedencia;\nb) Por inadimplemento de qualquer clausula contratual;\nc) Por razoes de interesse publico devidamente justificadas;\nd) Por caso fortuito ou forca maior que impossibilite a continuidade dos servicos;\ne) Por falencia, recuperacao judicial ou dissolucao da CONTRATADA.")

    _write_section(pdf, 'CLAUSULA 9a - DO FORO',
        f"Fica eleito o foro da Comarca de {EMPRESA_CIDADE}, Estado de {EMPRESA_ESTADO}, para dirimir quaisquer questoes oriundas do presente contrato, com renuncia expressa de qualquer outro, por mais privilegiado que seja.\n\nE, por estarem assim justas e contratadas, as partes assinam o presente instrumento em 2 (duas) vias de igual teor e forma, na presenca das testemunhas abaixo.\n\n{EMPRESA_CIDADE}/{EMPRESA_ESTADO_SIGLA}, {row['data_inicio']}.")

    # Assinaturas
    pdf.ln(10)
    pdf.set_x(10)
    pdf.set_font('Helvetica', '', 9)
    half = EFF_W / 2
    pdf.cell(half, 5, '____________________________________', align='C')
    pdf.cell(half, 5, '____________________________________', align='C', new_x='LMARGIN', new_y='NEXT')
    pdf.set_font('Helvetica', 'B', 9)
    pdf.cell(half, 5, _clean(EMPRESA_NOME_UPPER), align='C')
    pdf.cell(half, 5, _clean(contratada.upper()), align='C', new_x='LMARGIN', new_y='NEXT')
    pdf.set_font('Helvetica', '', 8)
    pdf.cell(half, 5, 'CONTRATANTE', align='C')
    pdf.cell(half, 5, 'CONTRATADA', align='C', new_x='LMARGIN', new_y='NEXT')

    pdf.ln(15)
    pdf.set_font('Helvetica', '', 8)
    pdf.cell(0, 5, 'Testemunhas:', new_x='LMARGIN', new_y='NEXT')
    pdf.ln(8)
    pdf.cell(half, 5, '____________________________________', align='C')
    pdf.cell(half, 5, '____________________________________', align='C', new_x='LMARGIN', new_y='NEXT')
    pdf.cell(half, 5, 'Nome:', align='C')
    pdf.cell(half, 5, 'Nome:', align='C', new_x='LMARGIN', new_y='NEXT')
    pdf.cell(half, 5, 'CPF:', align='C')
    pdf.cell(half, 5, 'CPF:', align='C', new_x='LMARGIN', new_y='NEXT')

    pdf.output(output_path)

print('Funcoes de geracao de contratos PDF definidas com sucesso.')

Funcoes de geracao de contratos PDF definidas com sucesso.


In [0]:
# Conteudo detalhado para cada procedimento
PROCEDIMENTOS_CONTEUDO = {
    'PROC-SMS-001_Permissao_Trabalho_Quente.pdf': {
        'titulo': 'Permissao de Trabalho a Quente (PTQ)',
        'codigo': 'PROC-SMS-001',
        'revisao': '05',
        'objetivo': 'Estabelecer os requisitos e criterios de seguranca para a emissao de Permissao de Trabalho a Quente (PTQ) em todas as unidades industriais da fake, visando prevenir incidentes com fogo e explosao durante atividades que envolvam fontes de ignicao.',
        'aplicabilidade': 'Este procedimento aplica-se a todas as atividades que envolvam chama aberta, soldagem, corte termico, esmerilhamento e quaisquer operacoes que possam gerar faiscas ou calor suficiente para ignizar atmosferas inflamaveis, em todas as unidades industriais da fake S.A.',
        'definicoes': [
            'Trabalho a Quente: Toda atividade que envolva chama aberta, geracao de faiscas ou temperaturas capazes de iniciar combustao em atmosferas inflamaveis.',
            'PTQ: Permissao de Trabalho a Quente - documento formal que autoriza a execucao de trabalho a quente apos verificacao das condicoes de seguranca.',
            'LEL: Lower Explosive Limit - Limite Inferior de Explosividade.',
            'Vigia de Fogo: Profissional designado para monitorar continuamente as condicoes de seguranca durante e apos o trabalho a quente.',
            'APR: Analise Preliminar de Riscos - documento que identifica perigos e medidas de controle.',
        ],
        'responsabilidades': [
            'Gerente de Area: Aprovar a PTQ e garantir que os recursos necessarios estejam disponiveis.',
            'Tecnico de Seguranca: Realizar inspecao previa da area, medir concentracao de gases e validar condicoes.',
            'Executor: Cumprir rigorosamente as condicoes estabelecidas na PTQ.',
            'Vigia de Fogo: Permanecer na area durante toda a execucao e 30 minutos apos o termino.',
            'Operador de Area: Informar sobre condicoes operacionais e riscos especificos da area.',
        ],
        'etapas': [
            '1. Solicitacao formal da PTQ pelo responsavel da atividade, com descricao detalhada do servico.',
            '2. Elaboracao da APR especifica para a atividade, com identificacao de todos os riscos.',
            '3. Inspecao da area pelo Tecnico de Seguranca, incluindo medicao de gases (LEL < 10%%).',
            '4. Verificacao do isolamento de fontes de energia e produto nas proximidades.',
            '5. Instalacao de biombos, mantas e barreiras contra projecao de faiscas.',
            '6. Disponibilizacao de extintores de incendio compativeis na area de trabalho.',
            '7. Designacao e posicionamento do Vigia de Fogo.',
            '8. Aprovacao e assinatura da PTQ pelo Gerente de Area.',
            '9. Comunicacao a sala de controle sobre o inicio do trabalho a quente.',
            '10. Monitoramento continuo durante a execucao e 30 minutos apos a conclusao.',
        ],
        'requisitos_seguranca': [
            'Medicao de atmosfera com explosimetro calibrado (LEL deve ser inferior a 10%%).',
            'Isolamento e purga de equipamentos e tubulacoes nas proximidades.',
            'Remocao de materiais inflamaveis no raio minimo de 15 metros.',
            'Utilizacao obrigatoria de EPI: mascara de solda, avental de raspa, luvas, perneiras.',
            'Comunicacao continua com a sala de controle operacional.',
            'Verificacao das condicoes climaticas (vento, umidade) antes do inicio.',
        ],
    },
    'PROC-SMS-002_Bloqueio_Etiquetagem_LOTO.pdf': {
        'titulo': 'Bloqueio e Etiquetagem (LOTO - Lockout/Tagout)',
        'codigo': 'PROC-SMS-002',
        'revisao': '07',
        'objetivo': 'Definir os requisitos para isolamento, bloqueio e etiquetagem de fontes de energia perigosa durante atividades de manutencao, inspecao e reparo em equipamentos e instalacoes industriais, prevenindo a energizacao ou movimentacao acidental.',
        'aplicabilidade': 'Aplicavel a todas as fontes de energia (eletrica, mecanica, hidraulica, pneumatica, termica, quimica e gravitacional) em todas as unidades industriais da fake S.A.',
        'definicoes': [
            'LOTO: Lockout/Tagout - Sistema de bloqueio e etiquetagem para isolamento de fontes de energia.',
            'Energia Zero: Condicao em que todas as fontes de energia perigosa estao isoladas e dissipadas.',
            'Dispositivo de Bloqueio: Cadeado, trava ou dispositivo mecanico que impede a operacao do equipamento.',
            'Etiqueta de Seguranca: Identificacao visual que indica a condicao de bloqueio e o responsavel.',
            'Caixa de Bloqueio Multiplo: Dispositivo que permite multiplos bloqueios simultaneos por diferentes executores.',
        ],
        'responsabilidades': [
            'Supervisor de Manutencao: Coordenar o processo de bloqueio e garantir que todos os pontos estejam isolados.',
            'Executor: Aplicar seu cadeado pessoal e verificar a condicao de energia zero antes de iniciar o trabalho.',
            'Operador de Painel: Confirmar o desligamento dos equipamentos na sala de controle.',
            'Tecnico de Seguranca: Auditar o processo de bloqueio e verificar conformidade.',
        ],
        'etapas': [
            '1. Identificar todas as fontes de energia associadas ao equipamento.',
            '2. Notificar todos os funcionarios afetados sobre o bloqueio planejado.',
            '3. Desligar o equipamento seguindo o procedimento operacional normal.',
            '4. Aplicar dispositivos de bloqueio em todos os pontos de isolamento.',
            '5. Fixar etiquetas de seguranca com identificacao do responsavel e motivo.',
            '6. Dissipar toda energia residual (pressao, tensao, temperatura).',
            '7. Verificar a condicao de energia zero por tentativa de acionamento.',
            '8. Cada executor aplica seu cadeado pessoal na caixa de bloqueio multiplo.',
            '9. Apos conclusao do servico, cada executor remove apenas seu proprio cadeado.',
            '10. O ultimo a remover verifica a area antes de autorizar a re-energizacao.',
        ],
        'requisitos_seguranca': [
            'Cada trabalhador deve possuir cadeado pessoal intransferivel com chave unica.',
            'Proibido remover bloqueio ou etiqueta de outro trabalhador.',
            'Bloqueio deve permanecer durante toda a duracao do servico, incluindo intervalos.',
            'Em troca de turno, o bloqueio deve ser transferido antes da saida do executor.',
            'Testes de energia zero devem ser repetidos apos qualquer interrupcao do servico.',
        ],
    },
    'PROC-SMS-003_Trabalho_em_Altura.pdf': {
        'titulo': 'Trabalho em Altura',
        'codigo': 'PROC-SMS-003',
        'revisao': '04',
        'objetivo': 'Estabelecer requisitos de seguranca para atividades realizadas acima de 2,0 metros do nivel inferior, onde haja risco de queda, conforme NR-35 do Ministerio do Trabalho.',
        'aplicabilidade': 'Todas as atividades em altura nas unidades industriais, incluindo manutencao de torres, vasos, caldeiras, estruturas metalicas, pipe racks e montagem de andaimes.',
        'definicoes': [
            'Trabalho em Altura: Toda atividade executada acima de 2,0 metros do nivel inferior.',
            'Cinto tipo paraquedista: EPI obrigatorio com talabarte duplo e absorvedor de energia.',
            'Ponto de Ancoragem: Ponto estrutural certificado para fixacao do sistema de protecao.',
            'Linha de Vida: Sistema horizontal ou vertical para movimentacao segura em altura.',
        ],
        'responsabilidades': [
            'Engenheiro de Seguranca: Definir pontos de ancoragem e sistemas de protecao.',
            'Executor: Possuir treinamento NR-35 valido e ASO para trabalho em altura.',
            'Supervisor: Inspecionar andaimes e plataformas antes de autorizar acesso.',
        ],
        'etapas': [
            '1. Verificar validade do treinamento NR-35 e ASO dos trabalhadores.',
            '2. Inspecionar todos os EPIs: cinto, talabarte, trava-quedas, capacete com jugular.',
            '3. Avaliar condicoes climaticas (ventos acima de 40 km/h suspendem atividades).',
            '4. Sinalizar e isolar a area abaixo do trabalho em altura.',
            '5. Verificar integridade de andaimes, plataformas e escadas.',
            '6. Conectar ao ponto de ancoragem antes de acessar a posicao elevada.',
            '7. Utilizar bolsa porta-ferramentas para evitar queda de objetos.',
            '8. Manter comunicacao constante com equipe de apoio no solo.',
        ],
        'requisitos_seguranca': [
            'Treinamento NR-35 com reciclagem a cada 2 anos.',
            'ASO especifico para trabalho em altura (sem restricoes medicas).',
            'Cinto tipo paraquedista com talabarte duplo e absorvedor de energia.',
            'Andaimes devem ter guarda-corpo, rodape e plataforma completa.',
            'Atividades suspensas em caso de chuva, neblina ou ventos fortes.',
        ],
    },
    'PROC-SMS-004_Espaco_Confinado.pdf': {
        'titulo': 'Entrada em Espaco Confinado',
        'codigo': 'PROC-SMS-004',
        'revisao': '06',
        'objetivo': 'Definir procedimentos seguros para entrada, permanencia e resgate em espacos confinados, conforme NR-33, em vasos de pressao, tanques, reatores, colunas e demais equipamentos das unidades petroquimicas.',
        'aplicabilidade': 'Todos os espacos confinados nas unidades industriais da fake, incluindo vasos de pressao, reatores de polimerizacao, colunas de destilacao, tanques de armazenamento e galerias subterraneas.',
        'definicoes': [
            'Espaco Confinado: Area nao projetada para ocupacao continua, com meios limitados de entrada/saida e risco de atmosfera perigosa.',
            'PET: Permissao de Entrada e Trabalho em Espaco Confinado.',
            'Vigia: Profissional treinado que permanece fora do espaco monitorando os trabalhadores.',
            'Equipe de Resgate: Equipe treinada e equipada para resgate em espaco confinado.',
        ],
        'responsabilidades': [
            'Supervisor de Entrada: Autorizar a entrada apos verificacao de todas as condicoes.',
            'Vigia: Manter contato visual/verbal continuo com os trabalhadores no interior.',
            'Entrantes: Utilizar todos os EPIs e equipamentos de monitoramento pessoal.',
            'Equipe de Resgate: Permanecer em prontidao durante toda a operacao.',
        ],
        'etapas': [
            '1. Emitir PET com identificacao de todos os riscos e medidas de controle.',
            '2. Isolar, bloquear e etiquetar todas as fontes de energia e produto (LOTO).',
            '3. Purgar e ventilar o espaco com ar fresco forcado.',
            '4. Monitorar atmosfera: O2 (19,5%%-23%%), LEL (<10%%), H2S (<10ppm), CO (<25ppm).',
            '5. Posicionar equipamento de resgate (tripe com guincho na boca de visita).',
            '6. Designar vigia e equipe de resgate em prontidao.',
            '7. Autorizar entrada com monitoramento continuo de atmosfera.',
            '8. Evacuar imediatamente em caso de alarme dos detectores.',
        ],
        'requisitos_seguranca': [
            'Monitoramento continuo com detector multigas pessoal calibrado.',
            'Ventilacao forcada durante toda a permanencia.',
            'Tripe de resgate instalado quando a entrada for vertical.',
            'Comunicacao por radio entre vigia, entrantes e sala de controle.',
            'Treinamento NR-33 valido para todos os envolvidos.',
        ],
    },
    'PROC-SMS-005_Manuseio_Produtos_Quimicos.pdf': {
        'titulo': 'Manuseio de Produtos Quimicos Perigosos',
        'codigo': 'PROC-SMS-005',
        'revisao': '03',
        'objetivo': 'Estabelecer diretrizes para o manuseio seguro de produtos quimicos perigosos utilizados nos processos petroquimicos, incluindo materias-primas, catalisadores, solventes e reagentes.',
        'aplicabilidade': 'Manuseio de todos os produtos quimicos classificados como perigosos (toxicos, inflamaveis, corrosivos, oxidantes) em todas as unidades da fake.',
        'definicoes': [
            'FISPQ: Ficha de Informacoes de Seguranca de Produtos Quimicos.',
            'GHS: Sistema Globalmente Harmonizado de Classificacao e Rotulagem.',
            'EPI Quimico: Equipamento de protecao especifico para contato com quimicos.',
            'Chuveiro de Emergencia: Equipamento para descontaminacao rapida em caso de contato.',
        ],
        'responsabilidades': [
            'Operador: Conhecer a FISPQ dos produtos manuseados em sua area.',
            'Tecnico Quimico: Classificar e rotular produtos conforme GHS.',
            'Seguranca do Trabalho: Garantir disponibilidade de EPI quimico e chuveiros.',
        ],
        'etapas': [
            '1. Consultar FISPQ antes de qualquer manuseio de produto quimico.',
            '2. Selecionar e vestir EPI adequado conforme classificacao do produto.',
            '3. Verificar integridade de embalagens, recipientes e conexoes.',
            '4. Realizar transferencia em area ventilada com sistema de contencao.',
            '5. Monitorar para deteccao de vazamentos durante todo o processo.',
            '6. Destinar residuos conforme Plano de Gerenciamento de Residuos.',
        ],
        'requisitos_seguranca': [
            'FISPQ disponivel em portugues no local de trabalho.',
            'Chuveiros de emergencia e lava-olhos testados semanalmente.',
            'Compatibilidade quimica verificada antes de armazenamento conjunto.',
            'Kit de emergencia para vazamentos (spill kit) disponivel na area.',
        ],
    },
    'PROC-ENG-001_Gestao_Mudancas_MOC.pdf': {
        'titulo': 'Gestao de Mudancas (Management of Change - MOC)',
        'codigo': 'PROC-ENG-001',
        'revisao': '04',
        'objetivo': 'Definir o processo formal para avaliacao, aprovacao e implementacao de mudancas em processos, equipamentos, procedimentos e organizacao, garantindo que os riscos sejam avaliados e controlados.',
        'aplicabilidade': 'Todas as mudancas em processos petroquimicos, equipamentos criticos, sistemas de controle, procedimentos operacionais e organizacao de pessoal nas unidades da fake.',
        'definicoes': [
            'MOC: Management of Change - processo formal de gestao de mudancas.',
            'Mudanca Temporaria: Alteracao com prazo definido que sera revertida.',
            'Mudanca Permanente: Alteracao definitiva em processos ou instalacoes.',
            'HAZOP: Hazard and Operability Study - estudo de perigos e operabilidade.',
        ],
        'responsabilidades': [
            'Solicitante: Descrever a mudanca e justificativa tecnica.',
            'Engenharia de Processos: Avaliar impactos no processo e seguranca.',
            'Comite de MOC: Aprovar ou rejeitar a mudanca com base na analise de riscos.',
            'SMS: Validar medidas de seguranca associadas a mudanca.',
        ],
        'etapas': [
            '1. Abertura do formulario MOC pelo solicitante com descricao detalhada.',
            '2. Classificacao da mudanca (temporaria/permanente, nivel de risco).',
            '3. Analise de riscos (HAZOP, What-If, ou APR conforme nivel).',
            '4. Avaliacao de impactos em seguranca, meio ambiente e operacao.',
            '5. Aprovacao pelo Comite de MOC com definicao de condicionantes.',
            '6. Implementacao da mudanca com acompanhamento tecnico.',
            '7. Atualizacao de documentos (P&IDs, procedimentos, treinamentos).',
            '8. Verificacao de eficacia e fechamento do MOC.',
        ],
        'requisitos_seguranca': [
            'Nenhuma mudanca pode ser implementada sem aprovacao formal do MOC.',
            'Mudancas emergenciais requerem ratificacao em ate 72 horas.',
            'Treinamento de todos os afetados antes da implementacao.',
            'Mudancas temporarias com prazo maximo de 90 dias.',
        ],
    },
    'PROC-ENG-002_Comissionamento_Startup.pdf': {
        'titulo': 'Comissionamento e Partida de Unidades (Startup)',
        'codigo': 'PROC-ENG-002',
        'revisao': '03',
        'objetivo': 'Estabelecer requisitos e etapas para o comissionamento e partida segura de unidades petroquimicas, incluindo testes de aceitacao, pre-operacao e operacao assistida.',
        'aplicabilidade': 'Todas as partidas de unidades novas, apos paradas programadas, ou retorno de grandes manutencoes nas unidades de polimerizacao, cracking e utilidades.',
        'definicoes': [
            'Comissionamento: Processo de verificacao e teste de sistemas antes da operacao.',
            'Pre-Startup Safety Review (PSSR): Revisao de seguranca antes da partida.',
            'Punch List: Lista de pendencias identificadas durante o comissionamento.',
        ],
        'responsabilidades': [
            'Lider de Comissionamento: Coordenar todas as atividades de pre-partida.',
            'Engenharia de Processos: Validar parametros operacionais e intertrips.',
            'Operacao: Participar dos testes funcionais e validacao de procedimentos.',
        ],
        'etapas': [
            '1. Verificacao mecanica de integridade (Mechanical Completion).',
            '2. Teste hidrostatico de vasos e tubulacoes conforme norma ASME.',
            '3. Flush e limpeza quimica de linhas de processo.',
            '4. Testes de malhas de instrumentacao e sistemas de controle.',
            '5. Teste de intertrips e sistemas de seguranca (SIS).',
            '6. PSSR - Revisao de Seguranca pre-partida.',
            '7. Inertizacao com nitrogenio e verificacao de estanqueidade.',
            '8. Introducao gradual de carga e rampa de producao.',
        ],
        'requisitos_seguranca': [
            'PSSR obrigatorio antes de qualquer partida.',
            'Todos os itens criticos do punch list devem estar resolvidos.',
            'Equipe de emergencia em prontidao durante a partida.',
            'Monitoramento 24h nas primeiras 72 horas de operacao.',
        ],
    },
    'PROC-MNT-001_Plano_Manutencao_Preventiva.pdf': {
        'titulo': 'Plano de Manutencao Preventiva',
        'codigo': 'PROC-MNT-001',
        'revisao': '08',
        'objetivo': 'Definir diretrizes para o planejamento, programacao e execucao da manutencao preventiva de equipamentos criticos das unidades petroquimicas, maximizando a disponibilidade operacional e a confiabilidade dos ativos.',
        'aplicabilidade': 'Todos os equipamentos classificados como criticos e semi-criticos, incluindo reatores, compressores, bombas, trocadores de calor, caldeiras e sistemas de utilidades.',
        'definicoes': [
            'Manutencao Preventiva: Intervencao programada baseada em tempo ou condicao.',
            'PCM: Planejamento e Controle de Manutencao.',
            'MTBF: Mean Time Between Failures - Tempo Medio Entre Falhas.',
            'MTTR: Mean Time To Repair - Tempo Medio de Reparo.',
            'Backlog: Acervo de ordens de servico pendentes de execucao.',
        ],
        'responsabilidades': [
            'PCM: Planejar e programar atividades preventivas conforme cronograma.',
            'Executores: Realizar as atividades conforme procedimento e registrar resultados.',
            'Engenharia de Manutencao: Analisar indicadores e otimizar planos.',
            'Operacao: Liberar equipamentos conforme programacao acordada.',
        ],
        'etapas': [
            '1. Classificacao de criticidade dos equipamentos (ABC).',
            '2. Definicao de planos de manutencao baseados em recomendacoes do fabricante e historico.',
            '3. Programacao semanal com otimizacao de recursos.',
            '4. Execucao das atividades com registro detalhado no SAP-PM.',
            '5. Inspecao de qualidade apos execucao.',
            '6. Analise de indicadores (MTBF, MTTR, aderencia ao plano).',
            '7. Revisao periodica dos planos com base em analise de falhas.',
        ],
        'requisitos_seguranca': [
            'Permissao de Trabalho obrigatoria para todas as intervencoes.',
            'LOTO deve ser aplicado antes de qualquer manutencao em equipamento energizado.',
            'Aderencia minima de 90%% ao plano de manutencao preventiva.',
            'Registros no sistema SAP-PM obrigatorios em ate 24h apos execucao.',
        ],
    },
    'PROC-MNT-002_Procedimento_Parada_Programada.pdf': {
        'titulo': 'Procedimento de Parada Programada (Turnaround)',
        'codigo': 'PROC-MNT-002',
        'revisao': '05',
        'objetivo': 'Estabelecer o processo de planejamento, execucao e encerramento de paradas programadas de manutencao (turnarounds) nas unidades petroquimicas.',
        'aplicabilidade': 'Todas as paradas programadas de unidades de producao, incluindo paradas gerais, parciais e de oportunidade.',
        'definicoes': [
            'Turnaround: Parada planejada para manutencao geral de uma unidade.',
            'Escopo Congelado: Lista definitiva de servicos aprovada para a parada.',
            'Caminho Critico: Sequencia de atividades que determina a duracao minima.',
        ],
        'responsabilidades': [
            'Gerente de Parada: Coordenacao geral e tomada de decisoes criticas.',
            'Planejador: Desenvolver cronograma detalhado e gerenciar escopo.',
            'Fiscal de Contrato: Acompanhar execucao dos contratados.',
        ],
        'etapas': [
            '1. Planejamento estrategico (12-18 meses antes): definicao de escopo preliminar.',
            '2. Planejamento detalhado (6-12 meses): cronograma, contratacoes, materiais.',
            '3. Pre-parada (3 meses): mobilizacao, canteiros, treinamentos.',
            '4. Congelamento de escopo (30 dias antes).',
            '5. Parada da unidade e descontaminacao.',
            '6. Execucao com acompanhamento diario de avanco e SMS.',
            '7. Comissionamento e partida (PSSR obrigatorio).',
            '8. Desmobilizacao e relatorio final de licoes aprendidas.',
        ],
        'requisitos_seguranca': [
            'Integracao de seguranca obrigatoria para todos os contratados.',
            'DDS (Dialogo Diario de Seguranca) antes de cada turno.',
            'Meta de zero acidentes com afastamento.',
            'Ambulatorio medico reforcado durante a parada.',
        ],
    },
    'PROC-MNT-003_Gestao_Sobressalentes.pdf': {
        'titulo': 'Gestao de Sobressalentes e Pecas de Reposicao',
        'codigo': 'PROC-MNT-003',
        'revisao': '03',
        'objetivo': 'Definir criterios para dimensionamento, aquisicao, armazenamento e controle de pecas sobressalentes e materiais de manutencao para equipamentos industriais.',
        'aplicabilidade': 'Todos os itens de estoque de manutencao, incluindo pecas de reposicao, consumiveis, lubrificantes e materiais auxiliares.',
        'definicoes': [
            'Sobressalente Critico: Peca cuja falta pode causar parada de producao.',
            'Ponto de Ressuprimento: Nivel minimo de estoque que dispara nova compra.',
            'Lead Time: Tempo entre pedido e entrega do fornecedor.',
        ],
        'responsabilidades': [
            'Almoxarifado: Controlar estoque e acuracidade de inventario.',
            'PCM: Definir lista de sobressalentes por equipamento.',
            'Compras: Negociar e formalizar contratos de fornecimento.',
        ],
        'etapas': [
            '1. Analise de criticidade e classificacao ABC de pecas.',
            '2. Definicao de estoque minimo e ponto de ressuprimento.',
            '3. Cadastro no sistema SAP-MM com especificacao tecnica completa.',
            '4. Armazenamento adequado conforme caracteristicas do material.',
            '5. Inventario ciclico com meta de acuracidade > 95%%.',
            '6. Analise de obsolescencia e descarte controlado.',
        ],
        'requisitos_seguranca': [
            'FISPQ disponivel para produtos quimicos armazenados.',
            'Compatibilidade quimica verificada no armazenamento.',
            'Rastreabilidade de materiais criticos de seguranca.',
        ],
    },
    'PROC-QLD-001_Controle_Qualidade_Resinas.pdf': {
        'titulo': 'Controle de Qualidade de Resinas Termoplasticas',
        'codigo': 'PROC-QLD-001',
        'revisao': '06',
        'objetivo': 'Estabelecer os ensaios, metodos analiticos e criterios de aceitacao para controle de qualidade das resinas termoplasticas produzidas (polietileno, polipropileno e PVC).',
        'aplicabilidade': 'Todas as linhas de producao de resinas termoplasticas nas unidades industriais, desde a polimerizacao ate a expedicao.',
        'definicoes': [
            'Indice de Fluidez (MFI): Medida da fluidez do polimero fundido, em g/10min.',
            'Densidade: Massa por unidade de volume da resina, em g/cm3.',
            'Cinzas: Residuo inorganico apos calcinacao da amostra.',
            'Gel Count: Contagem de particulas nao fundidas ou contaminantes.',
        ],
        'responsabilidades': [
            'Laboratorio: Realizar ensaios conforme metodos normatizados.',
            'Controle de Processos: Ajustar parametros com base nos resultados.',
            'Qualidade: Liberar ou bloquear lotes conforme especificacao.',
        ],
        'etapas': [
            '1. Coleta de amostras conforme plano de amostragem (a cada 2 horas ou por lote).',
            '2. Ensaio de Indice de Fluidez (ASTM D1238).',
            '3. Determinacao de Densidade (ASTM D1505).',
            '4. Ensaio de Resistencia ao Impacto (ASTM D256).',
            '5. Analise de Cinzas e Volateis.',
            '6. Inspecao visual de pellets (cor, forma, contaminacao).',
            '7. Emissao de Certificado de Qualidade para o lote.',
        ],
        'requisitos_seguranca': [
            'EPIs de laboratorio: oculos, luvas, jaleco.',
            'Capela de exaustao para ensaios com solventes.',
            'Equipamentos calibrados e rastreados conforme ISO 17025.',
        ],
    },
    'PROC-QLD-002_Especificacao_Tecnica_Materiais.pdf': {
        'titulo': 'Especificacao Tecnica de Materiais',
        'codigo': 'PROC-QLD-002',
        'revisao': '04',
        'objetivo': 'Definir criterios e requisitos tecnicos para especificacao, qualificacao e aceitacao de materiais adquiridos para uso nas unidades industriais.',
        'aplicabilidade': 'Todos os materiais de construcao, equipamentos, instrumentos e insumos adquiridos para as instalacoes industriais.',
        'definicoes': [
            'Datasheet: Folha de dados com especificacoes tecnicas do material.',
            'IQF: Indice de Qualificacao de Fornecedor.',
            'Inspecao de Recebimento: Verificacao do material no ato da entrega.',
        ],
        'responsabilidades': [
            'Engenharia: Elaborar datasheets e requisitos tecnicos.',
            'Qualidade: Inspecionar materiais no recebimento.',
            'Compras: Incluir requisitos tecnicos nos pedidos de compra.',
        ],
        'etapas': [
            '1. Elaboracao do datasheet com requisitos tecnicos detalhados.',
            '2. Selecao de fornecedores qualificados conforme IQF.',
            '3. Inclusao de requisitos tecnicos no processo de compra.',
            '4. Inspecao em fabrica (quando aplicavel).',
            '5. Inspecao de recebimento com verificacao documental e fisica.',
            '6. Liberacao ou devolucao do material.',
        ],
        'requisitos_seguranca': [
            'Certificados de material (mill certificates) obrigatorios.',
            'Rastreabilidade do material ate a corrida de fabricacao.',
            'Materiais criticos requerem inspecao em fabrica.',
        ],
    },
    'PROC-AMB-001_Gestao_Residuos_Industriais.pdf': {
        'titulo': 'Gestao de Residuos Industriais',
        'codigo': 'PROC-AMB-001',
        'revisao': '05',
        'objetivo': 'Estabelecer diretrizes para segregacao, acondicionamento, transporte e destinacao final de residuos solidos, liquidos e gasosos gerados nas unidades industriais, conforme legislacao ambiental vigente.',
        'aplicabilidade': 'Todos os residuos gerados nas unidades industriais, incluindo residuos de processo, manutencao, laboratorio e areas administrativas.',
        'definicoes': [
            'PGRS: Plano de Gerenciamento de Residuos Solidos.',
            'Classe I: Residuo perigoso (inflamavel, corrosivo, toxico, reativo).',
            'Classe II-A: Residuo nao perigoso, nao inerte.',
            'Classe II-B: Residuo nao perigoso, inerte.',
            'MTR: Manifesto de Transporte de Residuos.',
        ],
        'responsabilidades': [
            'Meio Ambiente: Gerenciar o PGRS e auditar processos de destinacao.',
            'Geracao: Segregar corretamente na origem conforme classificacao.',
            'Logistica: Transportar com documentacao ambiental completa (MTR).',
        ],
        'etapas': [
            '1. Classificacao dos residuos conforme NBR 10004.',
            '2. Segregacao na origem em recipientes identificados por cor.',
            '3. Acondicionamento temporario em area coberta e impermeabilizada.',
            '4. Emissao de MTR para cada embarque.',
            '5. Transporte por empresa licenciada com veiculo adequado.',
            '6. Destinacao final em unidade licenciada (reciclagem, coprocessamento, aterro).',
            '7. Arquivo de comprovantes por 5 anos.',
        ],
        'requisitos_seguranca': [
            'Segregacao obrigatoria na fonte geradora.',
            'Residuos Classe I em containeres estanques e identificados.',
            'Transportadora com licenca ambiental e seguro validos.',
            'Auditoria anual em destinadores finais.',
        ],
    },
    'PROC-AMB-002_Monitoramento_Emissoes.pdf': {
        'titulo': 'Monitoramento de Emissoes Atmosfericas',
        'codigo': 'PROC-AMB-002',
        'revisao': '04',
        'objetivo': 'Definir a sistematica de monitoramento de emissoes atmosfericas das unidades industriais, garantindo conformidade com licencas ambientais e padroes de qualidade do ar.',
        'aplicabilidade': 'Todas as fontes de emissao atmosferica, incluindo chamines, tochas, fugas em equipamentos e emissoes de VOCs (compostos organicos volateis).',
        'definicoes': [
            'CEMS: Continuous Emission Monitoring System - sistema de monitoramento continuo.',
            'VOC: Volatile Organic Compounds - compostos organicos volateis.',
            'LDAR: Leak Detection and Repair - programa de deteccao e reparo de vazamentos.',
            'Isocinesia: Condicao de amostragem onde a velocidade na sonda iguala a do fluxo.',
        ],
        'responsabilidades': [
            'Meio Ambiente: Coordenar campanhas de monitoramento e manter registros.',
            'Operacao: Manter equipamentos de combustao ajustados e eficientes.',
            'Manutencao: Executar LDAR conforme cronograma estabelecido.',
        ],
        'etapas': [
            '1. Inventariar todas as fontes de emissao da unidade.',
            '2. Instalar e calibrar CEMS em fontes criticas.',
            '3. Realizar campanhas de amostragem isocinetica semestral.',
            '4. Executar programa LDAR trimestral com detector FID/PID.',
            '5. Compilar inventario de emissoes de GEE (gases de efeito estufa).',
            '6. Reportar a agencia ambiental conforme licenca de operacao.',
        ],
        'requisitos_seguranca': [
            'CEMS calibrado conforme normas EPA/CETESB.',
            'Programa LDAR com meta de reducao de 2%% ao ano.',
            'Relatorio semestral ao orgao ambiental competente.',
        ],
    },
    'PROC-LOG-001_Transporte_Produtos_Perigosos.pdf': {
        'titulo': 'Transporte de Produtos Perigosos',
        'codigo': 'PROC-LOG-001',
        'revisao': '06',
        'objetivo': 'Definir requisitos para o transporte rodoviario de produtos quimicos perigosos, conforme Resolucao ANTT 5.947/2021 e NBR 14619.',
        'aplicabilidade': 'Transporte de todas as materias-primas, produtos acabados e residuos classificados como perigosos, por via rodoviaria.',
        'definicoes': [
            'Ficha de Emergencia: Documento com informacoes para atendimento emergencial.',
            'Envelope de Transporte: Conjunto de documentos obrigatorios para o transporte.',
            'Rotulo de Risco: Identificacao visual da classe de risco do produto.',
            'Painel de Seguranca: Placa com numero ONU e classe de risco.',
        ],
        'responsabilidades': [
            'Logistica: Preparar documentacao e inspecionar veiculos.',
            'Motorista: Portar documentacao completa e curso MOPP valido.',
            'Emergencia: Manter plano de atendimento a emergencias no transporte.',
        ],
        'etapas': [
            '1. Verificar classificacao ONU do produto e compatibilidade do veiculo.',
            '2. Preparar envelope de transporte (Ficha de Emergencia, FISPQ, NF).',
            '3. Inspecionar veiculo: sinalizacao, equipamentos de emergencia, condicoes mecanicas.',
            '4. Afixar rotulos de risco e paineis de seguranca.',
            '5. Realizar checklist de saida com motorista.',
            '6. Monitorar transporte via rastreamento GPS.',
            '7. Comunicar sala de emergencia sobre horario de saida e chegada prevista.',
        ],
        'requisitos_seguranca': [
            'Motorista com curso MOPP valido e CNH compativel.',
            'Veiculo com inspecao veicular semestral (INMETRO).',
            'Kit de emergencia conforme classe de risco do produto.',
            'Rastreamento GPS obrigatorio durante todo o percurso.',
        ],
    },
}


class ProcedimentoPDF(FPDF):
    def __init__(self, codigo='', titulo=''):
        super().__init__()
        self._proc_codigo = codigo
        self._proc_titulo = titulo

    def header(self):
        self.set_font('Helvetica', 'B', 9)
        self.cell(0, 5, 'EMPRESA FAKE S.A. - SISTEMA DE GESTAO INTEGRADA', align='C', new_x='LMARGIN', new_y='NEXT')
        self.set_font('Helvetica', '', 7)
        self.cell(0, 4, _clean(f'Codigo: {self._proc_codigo} | Classificacao: Uso Interno'), align='C', new_x='LMARGIN', new_y='NEXT')
        self.line(10, self.get_y() + 1, 200, self.get_y() + 1)
        self.ln(4)

    def footer(self):
        self.set_y(-15)
        self.set_font('Helvetica', 'I', 7)
        self.cell(0, 5, _clean(f'{self._proc_codigo} - {self._proc_titulo} | Pagina {self.page_no()}/{{nb}}'), align='C')


def gerar_procedimento_pdf(proc_filename, proc_data, output_path):
    pdf = ProcedimentoPDF(proc_data['codigo'], proc_data['titulo'])
    pdf.alias_nb_pages()
    pdf.set_auto_page_break(auto=True, margin=20)
    pdf.add_page()

    # Titulo
    pdf.set_font('Helvetica', 'B', 14)
    pdf.set_x(10)
    pdf.multi_cell(EFF_W, 7, _clean(proc_data['titulo'].upper()), align='C')
    pdf.ln(2)
    pdf.set_font('Helvetica', '', 9)
    pdf.cell(0, 5, _clean(f"Codigo: {proc_data['codigo']}  |  Revisao: {proc_data['revisao']}  |  Data: 15/01/2025"), align='C', new_x='LMARGIN', new_y='NEXT')
    pdf.cell(0, 5, _clean('Aprovado por: Gerencia de SMS / Gerencia de Engenharia'), align='C', new_x='LMARGIN', new_y='NEXT')
    pdf.ln(6)

    # Secoes textuais
    secs = [
        ('1. OBJETIVO', proc_data['objetivo']),
        ('2. APLICABILIDADE', proc_data['aplicabilidade']),
    ]
    for s_title, s_body in secs:
        pdf.set_x(10)
        pdf.set_font('Helvetica', 'B', 10)
        pdf.multi_cell(EFF_W, 7, _clean(s_title))
        pdf.set_font('Helvetica', '', 9)
        pdf.set_x(10)
        pdf.multi_cell(EFF_W, 5, _clean(s_body))
        pdf.ln(3)

    # Definicoes
    pdf.set_x(10)
    pdf.set_font('Helvetica', 'B', 10)
    pdf.multi_cell(EFF_W, 7, '3. DEFINICOES')
    pdf.set_font('Helvetica', '', 9)
    for d in proc_data['definicoes']:
        pdf.set_x(10)
        pdf.multi_cell(EFF_W, 5, _clean(f'  - {d}'))
        pdf.ln(1)
    pdf.ln(3)

    # Responsabilidades
    pdf.set_x(10)
    pdf.set_font('Helvetica', 'B', 10)
    pdf.multi_cell(EFF_W, 7, '4. RESPONSABILIDADES')
    pdf.set_font('Helvetica', '', 9)
    for r in proc_data['responsabilidades']:
        pdf.set_x(10)
        pdf.multi_cell(EFF_W, 5, _clean(f'  - {r}'))
        pdf.ln(1)
    pdf.ln(3)

    # Procedimento
    pdf.set_x(10)
    pdf.set_font('Helvetica', 'B', 10)
    pdf.multi_cell(EFF_W, 7, '5. PROCEDIMENTO')
    pdf.set_font('Helvetica', '', 9)
    for e in proc_data['etapas']:
        pdf.set_x(10)
        pdf.multi_cell(EFF_W, 5, _clean(f'  {e}'))
        pdf.ln(1)
    pdf.ln(3)

    # Requisitos de Seguranca
    pdf.set_x(10)
    pdf.set_font('Helvetica', 'B', 10)
    pdf.multi_cell(EFF_W, 7, '6. REQUISITOS DE SEGURANCA')
    pdf.set_font('Helvetica', '', 9)
    for req in proc_data['requisitos_seguranca']:
        pdf.set_x(10)
        pdf.multi_cell(EFF_W, 5, _clean(f'  - {req}'))
        pdf.ln(1)
    pdf.ln(3)

    # Referencias Normativas
    pdf.set_x(10)
    pdf.set_font('Helvetica', 'B', 10)
    pdf.multi_cell(EFF_W, 7, '7. REFERENCIAS NORMATIVAS')
    pdf.set_font('Helvetica', '', 9)
    refs = [
        'NR-10 a NR-35 - Normas Regulamentadoras do Ministerio do Trabalho',
        'NBR ISO 14001 - Sistema de Gestao Ambiental',
        'NBR ISO 45001 - Sistema de Gestao de Saude e Seguranca Ocupacional',
        'Politica de SMS da fake S.A.',
        'Manual do Sistema de Gestao Integrada - fake',
    ]
    for ref in refs:
        pdf.set_x(10)
        pdf.multi_cell(EFF_W, 5, _clean(f'  - {ref}'))
        pdf.ln(1)

    # Historico de Revisoes
    pdf.ln(5)
    pdf.set_x(10)
    pdf.set_font('Helvetica', 'B', 10)
    pdf.multi_cell(EFF_W, 7, '8. HISTORICO DE REVISOES')
    pdf.set_font('Helvetica', 'B', 8)
    pdf.cell(30, 6, 'Revisao', border=1, align='C')
    pdf.cell(30, 6, 'Data', border=1, align='C')
    pdf.cell(50, 6, 'Autor', border=1, align='C')
    pdf.cell(80, 6, 'Descricao', border=1, align='C', new_x='LMARGIN', new_y='NEXT')
    pdf.set_font('Helvetica', '', 8)
    rev_num = int(proc_data['revisao'])
    for i in range(max(1, rev_num - 2), rev_num + 1):
        pdf.cell(30, 6, f'{i:02d}', border=1, align='C')
        pdf.cell(30, 6, f'{15-i:02d}/{min(i+1,12):02d}/2024', border=1, align='C')
        pdf.cell(50, 6, 'Gerencia de SMS', border=1, align='C')
        desc = 'Emissao inicial' if i == 1 else 'Revisao geral do procedimento'
        pdf.cell(80, 6, desc, border=1, align='C', new_x='LMARGIN', new_y='NEXT')

    pdf.output(output_path)

print(f'Funcoes de geracao de procedimentos definidas.')
print(f'Procedimentos disponiveis: {len(PROCEDIMENTOS_CONTEUDO)}')

Funcoes de geracao de procedimentos definidas.
Procedimentos disponiveis: 15


In [0]:
# Substituir mencoes a empresas reais por dados ficticios gerados pelo Faker

def _anonymize_data(obj, replacements):
    """Percorre estrutura (dict/list/str) substituindo strings."""
    if isinstance(obj, str):
        for old, new in replacements.items():
            obj = obj.replace(old, new)
        return obj
    elif isinstance(obj, dict):
        return {k: _anonymize_data(v, replacements) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [_anonymize_data(item, replacements) for item in obj]
    return obj

# Ordem importa: substituir versoes mais longas antes das mais curtas
_empresa_replacements = {
    'fake S.A.': EMPRESA_NOME_COMPLETO,
    'fake': EMPRESA_NOME,
}

PROCEDIMENTOS_CONTEUDO = _anonymize_data(PROCEDIMENTOS_CONTEUDO, _empresa_replacements)

# Corrigir header da classe ProcedimentoPDF para usar nome ficticio
def _new_proc_header(self):
    self.set_font('Helvetica', 'B', 9)
    self.cell(0, 5, _clean(f'{EMPRESA_NOME_UPPER} - SISTEMA DE GESTAO INTEGRADA'), align='C', new_x='LMARGIN', new_y='NEXT')
    self.set_font('Helvetica', '', 7)
    self.cell(0, 4, _clean(f'Codigo: {self._proc_codigo} | Classificacao: Uso Interno'), align='C', new_x='LMARGIN', new_y='NEXT')
    self.line(10, self.get_y() + 1, 200, self.get_y() + 1)
    self.ln(4)

ProcedimentoPDF.header = _new_proc_header

# Redefinir gerar_procedimento_pdf para usar nome ficticio nas referencias normativas
def gerar_procedimento_pdf(proc_filename, proc_data, output_path):
    pdf = ProcedimentoPDF(proc_data['codigo'], proc_data['titulo'])
    pdf.alias_nb_pages()
    pdf.set_auto_page_break(auto=True, margin=20)
    pdf.add_page()

    # Titulo
    pdf.set_font('Helvetica', 'B', 14)
    pdf.set_x(10)
    pdf.multi_cell(EFF_W, 7, _clean(proc_data['titulo'].upper()), align='C')
    pdf.ln(2)
    pdf.set_font('Helvetica', '', 9)
    pdf.cell(0, 5, _clean(f"Codigo: {proc_data['codigo']}  |  Revisao: {proc_data['revisao']}  |  Data: 15/01/2025"), align='C', new_x='LMARGIN', new_y='NEXT')
    pdf.cell(0, 5, _clean('Aprovado por: Gerencia de SMS / Gerencia de Engenharia'), align='C', new_x='LMARGIN', new_y='NEXT')
    pdf.ln(6)

    # Secoes textuais
    secs = [
        ('1. OBJETIVO', proc_data['objetivo']),
        ('2. APLICABILIDADE', proc_data['aplicabilidade']),
    ]
    for s_title, s_body in secs:
        pdf.set_x(10)
        pdf.set_font('Helvetica', 'B', 10)
        pdf.multi_cell(EFF_W, 7, _clean(s_title))
        pdf.set_font('Helvetica', '', 9)
        pdf.set_x(10)
        pdf.multi_cell(EFF_W, 5, _clean(s_body))
        pdf.ln(3)

    # Definicoes
    pdf.set_x(10)
    pdf.set_font('Helvetica', 'B', 10)
    pdf.multi_cell(EFF_W, 7, '3. DEFINICOES')
    pdf.set_font('Helvetica', '', 9)
    for d in proc_data['definicoes']:
        pdf.set_x(10)
        pdf.multi_cell(EFF_W, 5, _clean(f'  - {d}'))
        pdf.ln(1)
    pdf.ln(3)

    # Responsabilidades
    pdf.set_x(10)
    pdf.set_font('Helvetica', 'B', 10)
    pdf.multi_cell(EFF_W, 7, '4. RESPONSABILIDADES')
    pdf.set_font('Helvetica', '', 9)
    for r in proc_data['responsabilidades']:
        pdf.set_x(10)
        pdf.multi_cell(EFF_W, 5, _clean(f'  - {r}'))
        pdf.ln(1)
    pdf.ln(3)

    # Procedimento
    pdf.set_x(10)
    pdf.set_font('Helvetica', 'B', 10)
    pdf.multi_cell(EFF_W, 7, '5. PROCEDIMENTO')
    pdf.set_font('Helvetica', '', 9)
    for e in proc_data['etapas']:
        pdf.set_x(10)
        pdf.multi_cell(EFF_W, 5, _clean(f'  {e}'))
        pdf.ln(1)
    pdf.ln(3)

    # Requisitos de Seguranca
    pdf.set_x(10)
    pdf.set_font('Helvetica', 'B', 10)
    pdf.multi_cell(EFF_W, 7, '6. REQUISITOS DE SEGURANCA')
    pdf.set_font('Helvetica', '', 9)
    for req in proc_data['requisitos_seguranca']:
        pdf.set_x(10)
        pdf.multi_cell(EFF_W, 5, _clean(f'  - {req}'))
        pdf.ln(1)
    pdf.ln(3)

    # Referencias Normativas - usando nome ficticio
    pdf.set_x(10)
    pdf.set_font('Helvetica', 'B', 10)
    pdf.multi_cell(EFF_W, 7, '7. REFERENCIAS NORMATIVAS')
    pdf.set_font('Helvetica', '', 9)
    refs = [
        'NR-10 a NR-35 - Normas Regulamentadoras do Ministerio do Trabalho',
        'NBR ISO 14001 - Sistema de Gestao Ambiental',
        'NBR ISO 45001 - Sistema de Gestao de Saude e Seguranca Ocupacional',
        f'Politica de SMS da {EMPRESA_NOME_COMPLETO}',
        f'Manual do Sistema de Gestao Integrada - {EMPRESA_NOME}',
    ]
    for ref in refs:
        pdf.set_x(10)
        pdf.multi_cell(EFF_W, 5, _clean(f'  - {ref}'))
        pdf.ln(1)

    # Historico de Revisoes
    pdf.ln(5)
    pdf.set_x(10)
    pdf.set_font('Helvetica', 'B', 10)
    pdf.multi_cell(EFF_W, 7, '8. HISTORICO DE REVISOES')
    pdf.set_font('Helvetica', 'B', 8)
    pdf.cell(30, 6, 'Revisao', border=1, align='C')
    pdf.cell(30, 6, 'Data', border=1, align='C')
    pdf.cell(50, 6, 'Autor', border=1, align='C')
    pdf.cell(80, 6, 'Descricao', border=1, align='C', new_x='LMARGIN', new_y='NEXT')
    pdf.set_font('Helvetica', '', 8)
    rev_num = int(proc_data['revisao'])
    for i in range(max(1, rev_num - 2), rev_num + 1):
        pdf.cell(30, 6, f'{i:02d}', border=1, align='C')
        pdf.cell(30, 6, f'{15-i:02d}/{min(i+1,12):02d}/2024', border=1, align='C')
        pdf.cell(50, 6, 'Gerencia de SMS', border=1, align='C')
        desc = 'Emissao inicial' if i == 1 else 'Revisao geral do procedimento'
        pdf.cell(80, 6, desc, border=1, align='C', new_x='LMARGIN', new_y='NEXT')

    pdf.output(output_path)

print(f'Anonimizacao concluida. Todas as referencias a empresas reais foram substituidas por: {EMPRESA_NOME_COMPLETO}')

Anonimizacao concluida. Todas as referencias a empresas reais foram substituidas por: Fake Company S.A.


In [0]:
import time

# Amostra representativa: ~3 contratos por tipo_contrato, cobrindo diferentes unidades
sample = (
    contratos_full
    .groupby('tipo_contrato', group_keys=False)
    .apply(lambda g: g.sample(n=min(3, len(g)), random_state=42))
    .reset_index(drop=True)
)
print(f"Amostra selecionada: {len(sample)} contratos de {contratos_full['tipo_contrato'].nunique()} tipos")

start = time.time()
errors = []
success_count = 0

for idx, row in sample.iterrows():
    filename = row['documento_contrato']
    output_path = os.path.join(pdf_contratos_dir, filename)
    try:
        gerar_contrato_pdf(row, output_path)
        success_count += 1
    except Exception as e:
        errors.append((filename, str(e)))

elapsed = time.time() - start
print(f"Contratos gerados com sucesso: {success_count}/{len(sample)}")
print(f"Tempo total: {elapsed:.1f} segundos")
if errors:
    print(f"\nErros ({len(errors)}):")
    for fname, err in errors[:5]:
        print(f"  {fname}: {err}")

/home/spark-809c618f-03f0-422f-9ea2-ca/.ipykernel/12758/command-7200355714525261-680360703:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(3, len(g)), random_state=42))


Amostra selecionada: 30 contratos de 10 tipos
Contratos gerados com sucesso: 30/30
Tempo total: 3.1 segundos


In [0]:
start = time.time()
proc_success = 0
proc_errors = []

for proc_filename, proc_data in PROCEDIMENTOS_CONTEUDO.items():
    output_path = os.path.join(pdf_procedimentos_dir, proc_filename)
    try:
        gerar_procedimento_pdf(proc_filename, proc_data, output_path)
        proc_success += 1
        print(f"  Gerado: {proc_filename}")
    except Exception as e:
        proc_errors.append((proc_filename, str(e)))
        print(f"  ERRO: {proc_filename} - {e}")

elapsed = time.time() - start
print(f"\nProcedimentos gerados com sucesso: {proc_success}/{len(PROCEDIMENTOS_CONTEUDO)}")
print(f"Tempo total: {elapsed:.1f} segundos")

  Gerado: PROC-SMS-001_Permissao_Trabalho_Quente.pdf
  Gerado: PROC-SMS-002_Bloqueio_Etiquetagem_LOTO.pdf
  Gerado: PROC-SMS-003_Trabalho_em_Altura.pdf
  Gerado: PROC-SMS-004_Espaco_Confinado.pdf
  Gerado: PROC-SMS-005_Manuseio_Produtos_Quimicos.pdf
  Gerado: PROC-ENG-001_Gestao_Mudancas_MOC.pdf
  Gerado: PROC-ENG-002_Comissionamento_Startup.pdf
  Gerado: PROC-MNT-001_Plano_Manutencao_Preventiva.pdf
  Gerado: PROC-MNT-002_Procedimento_Parada_Programada.pdf
  Gerado: PROC-MNT-003_Gestao_Sobressalentes.pdf
  Gerado: PROC-QLD-001_Controle_Qualidade_Resinas.pdf
  Gerado: PROC-QLD-002_Especificacao_Tecnica_Materiais.pdf
  Gerado: PROC-AMB-001_Gestao_Residuos_Industriais.pdf
  Gerado: PROC-AMB-002_Monitoramento_Emissoes.pdf
  Gerado: PROC-LOG-001_Transporte_Produtos_Perigosos.pdf

Procedimentos gerados com sucesso: 15/15
Tempo total: 1.2 segundos


In [0]:
contratos_files = [f for f in os.listdir(pdf_contratos_dir) if f.endswith('.pdf')]
proc_files = [f for f in os.listdir(pdf_procedimentos_dir) if f.endswith('.pdf')]

print(f"=== RESUMO DA GERAÇÃO DE PDFs ===")
print(f"\nDiretório de contratos: {pdf_contratos_dir}")
print(f"  PDFs de contratos gerados: {len(contratos_files)}")
if contratos_files:
    sizes = [os.path.getsize(os.path.join(pdf_contratos_dir, f)) for f in contratos_files]
    print(f"  Tamanho total: {sum(sizes)/1024/1024:.1f} MB")
    print(f"  Tamanho médio: {sum(sizes)/len(sizes)/1024:.1f} KB")
    print(f"  Exemplos: {contratos_files[:5]}")

print(f"\nDiretório de procedimentos: {pdf_procedimentos_dir}")
print(f"  PDFs de procedimentos gerados: {len(proc_files)}")
for f in sorted(proc_files):
    size = os.path.getsize(os.path.join(pdf_procedimentos_dir, f))
    print(f"    {f} ({size/1024:.1f} KB)")

print(f"\n Total de PDFs gerados: {len(contratos_files) + len(proc_files)}")

=== RESUMO DA GERAÇÃO DE PDFs ===

Diretório de contratos: /Workspace/Users/carolina.ferreira@databricks.com/pdfs_contratos
  PDFs de contratos gerados: 30
  Tamanho total: 0.2 MB
  Tamanho médio: 6.0 KB
  Exemplos: ['CONTRATO_CT-2022-00693.pdf', 'CONTRATO_CT-2024-00491.pdf', 'CONTRATO_CT-2025-00986.pdf', 'CONTRATO_CT-2021-00687.pdf', 'CONTRATO_CT-2022-00065.pdf']

Diretório de procedimentos: /Workspace/Users/carolina.ferreira@databricks.com/pdfs_procedimentos
  PDFs de procedimentos gerados: 15
    PROC-AMB-001_Gestao_Residuos_Industriais.pdf (3.5 KB)
    PROC-AMB-002_Monitoramento_Emissoes.pdf (3.5 KB)
    PROC-ENG-001_Gestao_Mudancas_MOC.pdf (3.7 KB)
    PROC-ENG-002_Comissionamento_Startup.pdf (3.6 KB)
    PROC-LOG-001_Transporte_Produtos_Perigosos.pdf (3.5 KB)
    PROC-MNT-001_Plano_Manutencao_Preventiva.pdf (3.7 KB)
    PROC-MNT-002_Procedimento_Parada_Programada.pdf (3.5 KB)
    PROC-MNT-003_Gestao_Sobressalentes.pdf (3.5 KB)
    PROC-QLD-001_Controle_Qualidade_Resinas.pdf (3.5 